<a href="https://colab.research.google.com/github/thisishasan/speech_processing/blob/main/assignment_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install opensmile

In [2]:
!pip -q install pyannote.audio

In [3]:
import torch

In [4]:
from pyannote.audio import Pipeline
from pyannote.audio.pipelines.utils.hook import ProgressHook

In [6]:
PYANNOTEAI_API_KEY = ''

In [8]:
pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-precision-2", token=PYANNOTEAI_API_KEY)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.yaml:   0%|          | 0.00/134 [00:00<?, ?B/s]

In [9]:
AUDIO_PATH = "/content/voice_assignment_01.wav"

In [10]:
diarization = pipeline(AUDIO_PATH)

/usr/local/lib/python3.12/dist-packages/pyannoteai/sdk/client.py:356: UserWarning: 
You are using pyannoteAI's temporary storage solution. Your file will be permanently deleted from our servers within 24hs. 
If you are running in production, we highly recommend to use your own storage to reduce network latency and obtain results faster. 
Please check our documentation at https://docs.pyannote.ai/ for more information.
  warnings.warn("""
/usr/local/lib/python3.12/dist-packages/pyannoteai/sdk/client.py:600: UserWarning: 
You are using periodic polling to retrieve results. 
If you are running in production, we highly recommend to setup a webhook server to obtain results faster, as soon as they are available. 
Please check our documentation at https://docs.pyannote.ai/ for more information.
  warnings.warn("""


In [11]:
segments = []
for turn, speaker in diarization.speaker_diarization:
    segments.append({
        "start": turn.start,
        "end": turn.end,
        "duration": turn.end - turn.start,
        "speaker": speaker
    })

In [12]:
import pandas as pd

In [13]:
df_segments = pd.DataFrame(segments)
df_segments

,start,end,duration,speaker
0,0.865,3.445,2.58,SPEAKER_00
1,3.705,5.225,1.52,SPEAKER_01
2,5.745,7.865,2.12,SPEAKER_00
3,8.305,11.325,3.02,SPEAKER_01
4,11.705,12.805,1.10,SPEAKER_00
5,12.905,13.765,0.86,SPEAKER_01


In [14]:
speaker_durations = df_segments.groupby("speaker")["duration"].sum()
speaker_durations

speaker
SPEAKER_00    5.8
SPEAKER_01    5.4
Name: duration, dtype: float64

In [15]:
target_speaker = speaker_durations.idxmax()
target_speaker

'SPEAKER_00'

In [16]:
speaker_rows = df_segments[df_segments["speaker"] == target_speaker]
speaker_rows

,start,end,duration,speaker
0,0.865,3.445,2.58,SPEAKER_00
2,5.745,7.865,2.12,SPEAKER_00
4,11.705,12.805,1.10,SPEAKER_00


In [17]:
import soundfile as sf

In [18]:
audio, sr = sf.read(AUDIO_PATH)

In [19]:
chunks = []
for _, row in speaker_rows.iterrows():
    start_sample = int(row["start"] * sr)
    end_sample = int(row["end"] * sr)
    chunks.append(audio[start_sample:end_sample])
chunks

[array([-0.1138916 , -0.10549927, -0.06451416, ...,  0.01477051,
         0.30014038,  0.55410767], shape=(113778,)),
 array([ 0.23373413,  0.22814941,  0.24960327, ..., -0.35882568,
        -0.28515625, -0.28244019], shape=(93492,)),
 array([-0.23937988, -0.18997192, -0.15744019, ..., -0.43136597,
        -0.44555664, -0.45263672], shape=(48510,))]

In [20]:
import numpy as np

In [21]:
speaker_audio = np.concatenate(chunks)
speaker_audio

array([-0.1138916 , -0.10549927, -0.06451416, ..., -0.43136597,
       -0.44555664, -0.45263672], shape=(255780,))

In [22]:
SPEAKER_FILE = "/content/speaker_segment.wav"

In [23]:
sf.write(SPEAKER_FILE, speaker_audio, sr)

In [24]:
import opensmile

In [25]:
smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.LowLevelDescriptors
)

In [26]:
mfcc_df = smile.process_file(SPEAKER_FILE)
mfcc_df

Loudness_sma3  \
file                         start                  end                                     
/content/speaker_segment.wav 0 days 00:00:00        0 days 00:00:00.020000       2.056229   
                             0 days 00:00:00.010000 0 days 00:00:00.030000       2.097409   
                             0 days 00:00:00.020000 0 days 00:00:00.040000       2.242006   
                             0 days 00:00:00.030000 0 days 00:00:00.050000       2.736976   
                             0 days 00:00:00.040000 0 days 00:00:00.060000       3.183196   
...                                                                                   ...   
                             0 days 00:00:05.710000 0 days 00:00:05.730000       5.029598   
                             0 days 00:00:05.720000 0 days 00:00:05.740000       4.784603   
                             0 days 00:00:05.730000 0 days 00:00:05.750000       4.559546   
                             0 days 00:00:05.740000 0 days 00:00:05.760000       4.120558   
                             0 days 00:00:05.750000 0 days 00:00:05.800000       3.719552   

                                                                            alphaRatio_sma3  \
file                         start                  end                                       
/content/speaker_segment.wav 0 days 00:00:00        0 days 00:00:00.020000       -20.833380   
                             0 days 00:00:00.010000 0 days 00:00:00.030000       -21.646463   
                             0 days 00:00:00.020000 0 days 00:00:00.040000       -19.025665   
                             0 days 00:00:00.030000 0 days 00:00:00.050000       -14.315442   
                             0 days 00:00:00.040000 0 days 00:00:00.060000       -10.201191   
...                                                                                     ...   
                             0 days 00:00:05.710000 0 days 00:00:05.730000        -7.723227   
                             0 days 00:00:05.720000 0 days 00:00:05.740000        -7.164737   
                             0 days 00:00:05.730000 0 days 00:00:05.750000        -7.435979   
                             0 days 00:00:05.740000 0 days 00:00:05.760000        -9.522986   
                             0 days 00:00:05.750000 0 days 00:00:05.800000       -12.583435   

                                                                            hammarbergIndex_sma3  \
file                         start                  end                                            
/content/speaker_segment.wav 0 days 00:00:00        0 days 00:00:00.020000             38.171185   
                             0 days 00:00:00.010000 0 days 00:00:00.030000             37.992310   
                             0 days 00:00:00.020000 0 days 00:00:00.040000             33.148617   
                             0 days 00:00:00.030000 0 days 00:00:00.050000             26.494452   
                             0 days 00:00:00.040000 0 days 00:00:00.060000             20.807608   
...                                                                                          ...   
                             0 days 00:00:05.710000 0 days 00:00:05.730000             14.310601   
                             0 days 00:00:05.720000 0 days 00:00:05.740000             13.108597   
                             0 days 00:00:05.730000 0 days 00:00:05.750000             13.122921   
                             0 days 00:00:05.740000 0 days 00:00:05.760000             16.357195   
                             0 days 00:00:05.750000 0 days 00:00:05.800000             19.398466   

                                                                            slope0-500_sma3  \
file                         start                  end                                       
/content/speaker_segment.wav 0 days 00:00:00        0 days 00:00:00.020000        -0.112284   
                             0 days 00:00:00